In [1]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time
import re
from pdfminer.high_level import extract_text
from PyPDF2 import PdfReader
from datetime import datetime
import pdfplumber
from collections import defaultdict
import logging
import shutil
import tempfile

# Configure logging and start timer
start = time.time()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
pd.set_option('display.max_rows', 500, 'display.max_columns', 500, 'display.max_colwidth', 1000)

# Setup directories
BASE_DIR = "pcb_financial_data"
ORIGINAL_DIR = os.path.join(BASE_DIR, "original")
CURRENT_DIR = os.path.join(BASE_DIR, "current")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

# Create subdirectories for categorized files
for subfolder in ['balance-sheet', 'income-statement']:
    os.makedirs(os.path.join(CURRENT_DIR, subfolder), exist_ok=True)
    print(f"📁 Created directory: {os.path.join(CURRENT_DIR, subfolder)}")

for dir_path in [BASE_DIR, ORIGINAL_DIR, OUTPUT_DIR]:
    os.makedirs(dir_path, exist_ok=True)
    print(f"📁 Created directory: {dir_path}")

# Log file for tracking processed files
PROCESSED_FILES_LOG = os.path.join(BASE_DIR, "processed_files.csv")

bank_url = 'https://www.procreditbank-kos.com/shq/per-ne/publikimet-financiare/raportet-tremujore/'
print(f"🔗 Target URL: {bank_url}")

# Headers for requests
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36'
}

def load_processed_files():
    """Load list of already processed files from CSV"""
    if os.path.exists(PROCESSED_FILES_LOG):
        df = pd.read_csv(PROCESSED_FILES_LOG)
        return set(df['file_name'].tolist())
    return set()

def save_processed_file(file_name, status=1, reporting_date=None, current_name=None, is_corrupt=0):
    """Save processed file to log"""
    processed = load_processed_files()
    if file_name not in processed:
        df = pd.DataFrame({
            'file_name': [file_name],
            'processed_date': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
            'status': [status],
            'reporting_date': [reporting_date],
            'current_name': [current_name],
            'is_corrupt': [is_corrupt]
        })
        if os.path.exists(PROCESSED_FILES_LOG):
            existing = pd.read_csv(PROCESSED_FILES_LOG)
            df = pd.concat([existing, df], ignore_index=True)
        df.to_csv(PROCESSED_FILES_LOG, index=False)

def scrape_pdf_links(url):
    """Scrape PDF links from the ProCredit page"""
    print(f"🔍 Attempting to connect to: {url}")
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        print(f"✅ Successfully connected! Status code: {response.status_code}")
        
        soup = BeautifulSoup(response.text, 'html.parser')
        pdf_links = []
        
        for link in soup.find_all('a', href=True):
            href = link['href']
            if href.lower().endswith('.pdf') and not href.lower().startswith('https://91.187.98.58'):
                full_url = urljoin(url, href)
                pdf_links.append(full_url)
                print(f"   📄 Found PDF: {os.path.basename(full_url)}")
        
        print(f"\n✅ Total PDFs found: {len(pdf_links)}")
        return pdf_links
    except requests.exceptions.RequestException as e:
        print(f"❌ Error connecting to website: {e}")
        return []

def download_pdf(url, local_dir):
    """Download PDF to local directory"""
    file_name = os.path.basename(url)
    local_path = os.path.join(local_dir, file_name)
    
    print(f"   ⬇️ Downloading: {file_name}")
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        
        with open(local_path, 'wb') as f:
            f.write(response.content)
        
        file_size = os.path.getsize(local_path) / 1024
        print(f"   ✅ Downloaded: {file_name} ({file_size:.1f} KB)")
        return local_path
    except Exception as e:
        print(f"   ❌ Failed to download {file_name}: {e}")
        return None

def extract_date(text, filename):
    """Extract a date string from text or filename"""
    # Try to find date pattern like 31.12.2023 or 31/12/2023
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', text)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    
    # Try filename
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', filename)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    
    return None

def extract_date_from_filename(filename):
    """Extract date from filename"""
    match = re.search(r'\d{2}\.\d{2}\.\d{4}', filename)
    if match:
        return match.group()
    return None

def download_all(force=False):
    """Download PDFs and categorize them locally"""
    processed = set() if force else load_processed_files()
    if force:
        print("⚠️ Force mode enabled: all files will be downloaded")
    
    print(f"🔍 Scraping PDF links from: {bank_url}")
    pdf_links = scrape_pdf_links(bank_url)
    
    if not pdf_links:
        print("❌ No PDF links found")
        return []
    
    new_files = []
    file_metadata = []

    print(f"\n📥 Starting download of {len(pdf_links)} PDFs...")
    
    for i, pdf_url in enumerate(pdf_links, 1):
        file_name = os.path.basename(pdf_url)
        print(f"\n[{i}/{len(pdf_links)}] Processing: {file_name}")
        
        # Skip if already processed
        if file_name in processed and not force:
            print(f"   ⏭️ File already processed. Skipping...")
            continue

        new_files.append(file_name)
        
        # Download to original directory
        file_path = download_pdf(pdf_url, ORIGINAL_DIR)
        if not file_path:
            continue

        # Try to read metadata creation date
        dt = None
        try:
            with open(file_path, 'rb') as f:
                pdf_reader = PdfReader(f)
                metadata = pdf_reader.metadata
                if metadata:
                    for k, v in metadata.items():
                        if "/CreationDate" in k or "/ModDate" in k:
                            date_str = re.search(r'D:(\d+)', str(v))
                            if date_str:
                                dt = datetime.strptime(date_str.group(1), '%Y%m%d%H%M%S')
                                print(f"   📅 PDF creation date: {dt.date()}")
                                break
        except Exception as e:
            print(f"   ⚠️ Could not read metadata: {e}")

        # Extract date from PDF content
        try:
            text = extract_text(file_path)
            date_match = re.findall(r'\d{2}.\d{4}\b', text)
            if date_match:
                date_part = date_match[0]
                # Format date (add day based on month)
                if date_part[:2] == '12' or date_part[:2] == '03':
                    date = '31.' + date_part
                else:
                    date = '30.' + date_part
                print(f"   📅 Extracted reporting date: {date}")
            else:
                # Try full date
                date = extract_date(text, file_name)
                if date:
                    print(f"   📅 Extracted reporting date: {date}")
                else:
                    date = datetime.now().strftime('%d.%m.%Y')
                    print(f"   ⚠️ Could not extract date, using current: {date}")
        except Exception as e:
            print(f"   ⚠️ Error extracting text: {e}")
            date = datetime.now().strftime('%d.%m.%Y')

        # Create both BS and IS from the same PDF
        bs_new_name = f'pcb_bs_{date}.pdf'
        is_new_name = f'pcb_is_{date}.pdf'
        
        # Copy to both categorized folders
        shutil.copy2(file_path, os.path.join(CURRENT_DIR, 'balance-sheet', bs_new_name))
        shutil.copy2(file_path, os.path.join(CURRENT_DIR, 'income-statement', is_new_name))
        
        file_metadata.append({
            'file_name': file_name,
            'file_path': file_path,
            'bs_current_name': bs_new_name,
            'is_current_name': is_new_name,
            'reporting_date': date,
            'creation_date': dt,
            'download_date': datetime.now(),
            'is_corrupt': 0
        })
        
        print(f"   ✅ Created: {bs_new_name} and {is_new_name}")
        
        # Mark as processed
        save_processed_file(file_name, reporting_date=date, current_name=f"{bs_new_name}|{is_new_name}")

    # Save metadata
    if file_metadata:
        metadata_df = pd.DataFrame(file_metadata)
        metadata_path = os.path.join(OUTPUT_DIR, f"file_metadata_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        metadata_df.to_csv(metadata_path, index=False)
        print(f"\n💾 File metadata saved to {metadata_path}")

    if new_files:
        print(f"\n✅ Downloaded {len(new_files)} new files.")
    else:
        print("\n✅ No new files detected.")

    # Return list of all new current files
    current_files = []
    for m in file_metadata:
        current_files.append(m['bs_current_name'])
        current_files.append(m['is_current_name'])
    return current_files

def replace_negatives(value):
    """Convert parenthesized numbers to negative"""
    if pd.isna(value) or str(value).strip() in ['', '-', '0']:
        return '0'
    val = str(value).strip()
    if '(' in val and ')' in val:
        num = val.replace('(', '').replace(')', '').replace(',', '').strip()
        return f"-{num}"
    return val

def extract_values(line):
    """Extract numeric values from a line"""
    return re.findall(r'\(?[-\d,]+(?:\.\d+)?\)?', line)

def has_values(line):
    """Check if line contains numeric values"""
    return bool(re.search(r'\d', line))

# Data storage
income_data = defaultdict(list)
balance_data = defaultdict(list)

def is_duplicate(dt_report, category):
    """Check if category already exists for this date"""
    return any(entry.get('Category') == category for entry in income_data.get(dt_report, []))

def process_income_pdf(pdf_path, target_categories, filename):
    """Process income statement PDF (usually page 2)"""
    file_current_name = os.path.basename(filename)
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            if len(pdf.pages) > 1:
                page = pdf.pages[1]  # Income statement is on page 2
                text = page.extract_text()
                dt_report = extract_date(text, pdf_path)
                
                if dt_report:
                    # Convert to consistent format
                    date_obj = datetime.strptime(dt_report, '%d.%m.%Y')
                    dt_report_formatted = date_obj.strftime('%Y-%m-%d')
                else:
                    dt_report_formatted = datetime.now().strftime('%Y-%m-%d')
                
                lines = text.split('\n')
                category_index = 0
                start_storing = False
                
                for line in lines:
                    line = line.strip()
                    values = extract_values(line)
                    
                    if not start_storing:
                        if any(cat in line for cat in target_categories):
                            start_storing = True
                    
                    if start_storing and values:
                        # Determine category
                        if category_index < len(target_categories) and not any(cat in line for cat in target_categories):
                            category = target_categories[category_index]
                            category_index += 1
                        elif any(cat in line for cat in target_categories):
                            category = next((cat for cat in target_categories if cat in line), None)
                            if category:
                                category_index = target_categories.index(category) + 1
                        else:
                            category = None
                        
                        if category:
                            curr_quarter = values[-1] if values else "0"
                            prev_quarter = values[-2] if len(values) > 1 else "0"
                            
                            # Check for duplicates
                            if not any(entry.get('Category') == category for entry in income_data[dt_report_formatted]):
                                income_data[dt_report_formatted].append({
                                    'Category': category,
                                    'PREVIOUS_QUARTER': prev_quarter,
                                    'CURRENT_QUARTER': curr_quarter,
                                    'DT_REPORT': dt_report_formatted,
                                    'FILE_NAME': file_current_name
                                })
                                print(f"      ✅ Found income: {category}")
        
        print(f"   ✅ Processed income statement: {file_current_name}")
    except Exception as e:
        print(f"   ❌ Error processing income statement: {e}")

def process_balance_pdf(pdf_path, target_categories, filename):
    """Process balance sheet PDF"""
    file_current_name = os.path.basename(filename)
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            full_text = "".join(page.extract_text() or "" for page in pdf.pages)
            dt_report = extract_date(full_text, pdf_path)
            
            if dt_report:
                # Convert to consistent format
                date_obj = datetime.strptime(dt_report, '%d.%m.%Y')
                dt_report_formatted = date_obj.strftime('%Y-%m-%d')
            else:
                dt_report_formatted = datetime.now().strftime('%Y-%m-%d')
            
            lines = full_text.split('\n')
            for i, line in enumerate(lines):
                line = line.strip()
                if not line:
                    continue
                
                next_line = lines[i + 1].strip() if i + 1 < len(lines) else ""
                
                for category in target_categories:
                    if category.lower() in line.lower():
                        # Check for duplicates
                        if any(entry.get('Category') == category for entry in balance_data.get(dt_report_formatted, [])):
                            continue
                        
                        values = []
                        if has_values(line):
                            values = extract_values(line)
                        elif has_values(next_line) and not any(cat.lower() in next_line.lower() for cat in target_categories):
                            values = extract_values(next_line)
                        
                        curr_quarter = values[-1] if values else "0"
                        prev_quarter = values[-2] if len(values) > 1 else "0"
                        
                        balance_data[dt_report_formatted].append({
                            'Category': category,
                            'PREVIOUS_QUARTER': prev_quarter,
                            'CURRENT_QUARTER': curr_quarter,
                            'DT_REPORT': dt_report_formatted,
                            'FILE_NAME': file_current_name
                        })
                        print(f"      ✅ Found balance: {category}")
        
        print(f"   ✅ Processed balance sheet: {file_current_name}")
    except Exception as e:
        print(f"   ❌ Error processing balance sheet: {e}")

# Define categories
income_statement_categories = [
    'Të hyrat nga interesi', 'Shpenzimet e interesit', 'Neto të hyrat nga interesi',
    'Të hyrat nga tarifat dhe komisionet', 'Shpenzimet e tarifave dhe komisioneve',
    'Neto të hyrat nga tarifat dhe komisionet', 'Neto të hyrat nga tregtimi',
    'Neto të hyrat nga instrumentet tjera financiare',
    'Neto të hyrat (shpenzimet) tjera operative', 'Gjithsej të hyrat',
    'Provizionet për humbjet nga kreditë', 'Fitimi/(humbja) para tatimit',
    'Shpenzimet e tatimit në fitim', 'Fitimi/(humbja) neto',
    'Të ardhurat tjera gjithëpërfshirëse', 'Gjithsej të ardhurat gjithëpërfshirëse'
]

balance_sheet_categories = [
    "Paraja e gatshme dhe gjendja me BQK-në", "Kërkesat ndaj bankave", "Bonot e thesarit",
    "Investimet në letra me vlerë", 'Kreditë dhe paradhëniet ndaj klientëve',
    "Patundshmëritë dhe pajisjet", "Pasuritë e paprekshme", "Pasuritë tatimore të shtyra",
    "Pasuritë tjera", "Gjithsej pasuritë", "Depozitat e klientëve", "Detyrimet ndaj bankave",
    "Fondet tjera të huazuara", "Detyrimet tatimore të shtyra", "Detyrimet tjera",
    "Gjithsej detyrimet", "Kapitali aksionar", 'Rezervat e kapitalit',
    "Fitimi i mbajtur/(humbja) nga vitet paraprake", "Fitimi/(humbja) e vitit aktual",
    "Përbërësit tjerë të ekuitetit", 'Gjithsej ekuiteti i aksionarëve',
    "Gjithsej detyrimet dhe ekuiteti i aksionarëve"
]

def clean_numeric_values(df, prev_col, curr_col):
    """Clean and convert numeric columns"""
    for col in [prev_col, curr_col]:
        df[col] = df[col].astype(str).str.replace('-', '0', regex=False)
        df[col] = df[col].apply(replace_negatives)
        df[col] = df[col].str.replace(r'[^\d-]', '', regex=True)
        df[col] = pd.to_numeric(df[col].str.replace(',', ''), errors='coerce').fillna(0)
    return df

def main(force=False, extract_only=False):
    """Main execution function that returns unified DataFrames"""
    
    print("\n" + "="*60)
    print("🏦 PROCREDIT BANK FINANCIAL DATA EXTRACTION")
    print("="*60)
    
    # Clear previous data
    global income_data, balance_data
    income_data.clear()
    balance_data.clear()
    
    # Decide which files to process
    if extract_only:
        print("\n📂 EXTRACT-ONLY MODE: Processing existing files")
        new_files = []
        for category in ['balance-sheet', 'income-statement']:
            category_dir = os.path.join(CURRENT_DIR, category)
            if os.path.exists(category_dir):
                category_files = [f for f in os.listdir(category_dir) if f.startswith('pcb_') and f.endswith('.pdf')]
                new_files.extend(category_files)
        print(f"Found {len(new_files)} existing files")
    else:
        print("\n🌐 DOWNLOAD MODE: Downloading and processing new files")
        new_files = download_all(force=force)
    
    if new_files or extract_only:
        # Process balance sheet files
        bs_dir = os.path.join(CURRENT_DIR, 'balance-sheet')
        if os.path.exists(bs_dir):
            print(f"\n📂 Processing balance sheet files...")
            for filename in os.listdir(bs_dir):
                if filename.startswith('pcb_bs_') and filename.endswith('.pdf'):
                    if filename in new_files or extract_only:
                        file_path = os.path.join(bs_dir, filename)
                        print(f"\n   🔍 Processing: {filename}")
                        process_balance_pdf(file_path, balance_sheet_categories, filename)
        
        # Process income statement files
        is_dir = os.path.join(CURRENT_DIR, 'income-statement')
        if os.path.exists(is_dir):
            print(f"\n📂 Processing income statement files...")
            for filename in os.listdir(is_dir):
                if filename.startswith('pcb_is_') and filename.endswith('.pdf'):
                    if filename in new_files or extract_only:
                        file_path = os.path.join(is_dir, filename)
                        print(f"\n   🔍 Processing: {filename}")
                        process_income_pdf(file_path, income_statement_categories, filename)
        
        # Create DataFrames
        print("\n📊 Creating DataFrames...")
        
        # Balance Sheet DataFrame
        full_bs = pd.DataFrame()
        for date, data in balance_data.items():
            df = pd.DataFrame(data)
            full_bs = pd.concat([full_bs, df], ignore_index=True)
        
        if not full_bs.empty:
            full_bs.columns = ['BALANCE_CATEGORY', 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE', 'DT_REPORT', 'FILE_NAME']
            full_bs = clean_numeric_values(full_bs, 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE')
            
            # Category mapping for balance sheet
            bs_map = {
                'Paraja e gatshme dhe gjendja me BQK-në': '20',
                'Kërkesat ndaj bankave': '21', 
                'Bonot e thesarit': '22',
                'Investimet në letra me vlerë': '23',
                'Kreditë dhe paradhëniet ndaj klientëve': '26',
                'Patundshmëritë dhe pajisjet': '27',
                'Pasuritë e paprekshme': '28',
                'Pasuritë tatimore të shtyra': '29', 
                'Pasuritë tjera': '30',
                'Gjithsej pasuritë': '31',
                'Depozitat e klientëve': '32', 
                'Detyrimet ndaj bankave': '33', 
                'Fondet tjera të huazuara': '34',
                'Detyrimet tatimore të shtyra': '35', 
                'Detyrimet tjera': '36', 
                'Gjithsej detyrimet': '37',
                'Kapitali aksionar': '38',
                'Rezervat e kapitalit': '40', 
                'Fitimi i mbajtur/(humbja) nga vitet paraprake': '41',
                'Fitimi/(humbja) e vitit aktual': '42', 
                'Përbërësit tjerë të ekuitetit': '43',
                'Gjithsej ekuiteti i aksionarëve': '44',
                'Gjithsej detyrimet dhe ekuiteti i aksionarëve': '45'
            }
            
            full_bs['CATEGORY_CODE'] = full_bs['BALANCE_CATEGORY'].map(bs_map)
            full_bs['BANK_ID'] = 2  # ProCredit Bank ID
            full_bs['STATEMENT_TYPE'] = 'BALANCE_SHEET'
            full_bs['DT_REPORT'] = pd.to_datetime(full_bs['DT_REPORT'])
        
        # Income Statement DataFrame
        full_is = pd.DataFrame()
        for date, data in income_data.items():
            df = pd.DataFrame(data)
            full_is = pd.concat([full_is, df], ignore_index=True)
        
        if not full_is.empty:
            full_is.columns = ['INCOME_CATEGORY', 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE', 'DT_REPORT', 'FILE_NAME']
            full_is = clean_numeric_values(full_is, 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE')
            
            # Category mapping for income statement
            is_map = {
                'Të hyrat nga interesi': '1', 
                'Shpenzimet e interesit': '2', 
                'Neto të hyrat nga interesi': '3',
                'Të hyrat nga tarifat dhe komisionet': '4', 
                'Shpenzimet e tarifave dhe komisioneve': '5',
                'Neto të hyrat nga tarifat dhe komisionet': '6',
                'Neto të hyrat nga tregtimi': '7', 
                'Neto të hyrat nga instrumentet tjera financiare': '8',
                'Neto të hyrat (shpenzimet) tjera operative': '9', 
                'Gjithsej të hyrat': '10',
                'Provizionet për humbjet nga kreditë': '11',
                'Fitimi/(humbja) para tatimit': '14',
                'Shpenzimet e tatimit në fitim': '15',
                'Fitimi/(humbja) neto': '16',
                'Të ardhurat tjera gjithëpërfshirëse': '17', 
                'Gjithsej të ardhurat gjithëpërfshirëse': '19'
            }
            
            full_is['CATEGORY_CODE'] = full_is['INCOME_CATEGORY'].map(is_map)
            full_is['BANK_ID'] = 2  # ProCredit Bank ID
            full_is['STATEMENT_TYPE'] = 'INCOME_STATEMENT'
            full_is['DT_REPORT'] = pd.to_datetime(full_is['DT_REPORT'])
        
        # Create unified DataFrame
        unified_dfs = []
        
        if not full_is.empty:
            is_unified = full_is.rename(columns={
                'INCOME_CATEGORY': 'ORIGINAL_CATEGORY',
                'PREVIOUS_INCOME_VALUE': 'PREVIOUS_VALUE',
                'CURRENT_INCOME_VALUE': 'CURRENT_VALUE'
            })
            is_unified['CATEGORY_TYPE'] = 'INCOME'
            unified_dfs.append(is_unified)
        
        if not full_bs.empty:
            bs_unified = full_bs.rename(columns={
                'BALANCE_CATEGORY': 'ORIGINAL_CATEGORY',
                'PREVIOUS_BALANCE_VALUE': 'PREVIOUS_VALUE',
                'CURRENT_BALANCE_VALUE': 'CURRENT_VALUE'
            })
            bs_unified['CATEGORY_TYPE'] = 'BALANCE'
            unified_dfs.append(bs_unified)
        
        if unified_dfs:
            final_df = pd.concat(unified_dfs, ignore_index=True)
            final_df['CURR_ID'] = 1
            final_df['DATA_SOURCE'] = 'ProCredit Bank'
            final_df['EXTRACTION_DATE'] = datetime.now().strftime('%Y-%m-%d')
            final_df['REPORT_DATE'] = final_df['DT_REPORT'].dt.strftime('%Y-%m-%d')
            
            # Drop rows with invalid dates or categories
            final_df = final_df.dropna(subset=['DT_REPORT', 'CATEGORY_CODE'])
            
            # Sort
            final_df = final_df.sort_values(['DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_CODE'])
            
            # Reorder columns
            column_order = ['BANK_ID', 'REPORT_DATE', 'DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_TYPE',
                          'CATEGORY_CODE', 'ORIGINAL_CATEGORY', 'PREVIOUS_VALUE', 'CURRENT_VALUE',
                          'CURR_ID', 'FILE_NAME', 'DATA_SOURCE', 'EXTRACTION_DATE']
            
            final_columns = [col for col in column_order if col in final_df.columns]
            final_df = final_df[final_columns]
            
            # Save outputs
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            
            # Save unified DataFrame
            unified_path = os.path.join(OUTPUT_DIR, f"pcb_unified_financial_data_{timestamp}.csv")
            final_df.to_csv(unified_path, index=False)
            print(f"\n💾 Unified data saved to: {unified_path}")
            
            # Save individual files
            if not full_is.empty:
                is_path = os.path.join(OUTPUT_DIR, f"pcb_income_statement_{timestamp}.csv")
                full_is.to_csv(is_path, index=False)
            
            if not full_bs.empty:
                bs_path = os.path.join(OUTPUT_DIR, f"pcb_balance_sheet_{timestamp}.csv")
                full_bs.to_csv(bs_path, index=False)
            
            # Save Excel with multiple sheets
            excel_path = os.path.join(OUTPUT_DIR, f"pcb_financial_report_{timestamp}.xlsx")
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                final_df.to_excel(writer, sheet_name='Unified_Data', index=False)
                if not full_is.empty:
                    full_is.to_excel(writer, sheet_name='Income_Statement', index=False)
                if not full_bs.empty:
                    full_bs.to_excel(writer, sheet_name='Balance_Sheet', index=False)
            
            print(f"💾 Excel report saved to: {excel_path}")
            
            print(f"\n📊 UNIFIED DATAFRAME CREATED:")
            print(f"   - Total rows: {len(final_df)}")
            print(f"   - Income Statement rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'INCOME_STATEMENT'])}")
            print(f"   - Balance Sheet rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'BALANCE_SHEET'])}")
            print(f"   - Unique report dates: {final_df['REPORT_DATE'].nunique()}")
            print(f"   - Date range: {final_df['REPORT_DATE'].min()} to {final_df['REPORT_DATE'].max()}")
            
            elapsed_time = time.time() - start
            print(f"\n⏱️ Processing completed in {elapsed_time:.2f} seconds")
            
            return final_df, full_is, full_bs
        
        return pd.DataFrame(), full_is, full_bs
    
    return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

# Run the extraction
print("🚀 Starting ProCredit Bank financial data extraction...")
final_df, income_df, balance_df = main(force=True, extract_only=False)

if not final_df.empty:
    print("\n" + "="*60)
    print("📊 FINAL UNIFIED DATAFRAME")
    print("="*60)
    print(f"Shape: {final_df.shape}")
    print("\nFirst 10 rows:")
    print(final_df.head(10))
    
    print("\n📊 Summary by Statement Type:")
    print(final_df['STATEMENT_TYPE'].value_counts())
    
    print("\n📅 Reports by Date:")
    print(final_df['REPORT_DATE'].value_counts().sort_index())
    
    print("\n📊 Data Types:")
    print(final_df.dtypes)
else:
    print("❌ No data in final DataFrame")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


📁 Created directory: pcb_financial_data/current/balance-sheet
📁 Created directory: pcb_financial_data/current/income-statement
📁 Created directory: pcb_financial_data
📁 Created directory: pcb_financial_data/original
📁 Created directory: pcb_financial_data/output
🔗 Target URL: https://www.procreditbank-kos.com/shq/per-ne/publikimet-financiare/raportet-tremujore/
🚀 Starting ProCredit Bank financial data extraction...

🏦 PROCREDIT BANK FINANCIAL DATA EXTRACTION

🌐 DOWNLOAD MODE: Downloading and processing new files
⚠️ Force mode enabled: all files will be downloaded
🔍 Scraping PDF links from: https://www.procreditbank-kos.com/shq/per-ne/publikimet-financiare/raportet-tremujore/
🔍 Attempting to connect to: https://www.procreditbank-kos.com/shq/per-ne/publikimet-financiare/raportet-tremujore/
✅ Successfully connected! Status code: 200
   📄 Found PDF: 0001-Pasqyrat_Publikim_31.12.2025_gK8SA9GVT9_bNkYrJK3TW.pdf
   📄 Found PDF: 01-PasqyratperPublikim_[30.09.2025]_mvVADRNkNU_UEy8Fg9zzD.pdf
   📄

In [3]:
final_df.to_csv("PCB.csv")